In [2]:
# SoRL in modded-gpt compatible fashion (for ultra fast pre-training)
# 1. pre-training demands simple model architecture, even .generate function can be wrapped around the trained model afterwards
# 2. no need to include 'kv-cache' for the pre-training experiment here

In [ ]:
from sorl.model import CausalSelfAttention, Block, GPTConfig
import torch 

# mock input 
x = torch.randn(2, 1024, 768)

config = GPTConfig()
attn = CausalSelfAttention(dim=768, n_head=6)
block = Block(config=config)

y, v1 = attn(x)
x, v1 = block(x, v1, x, None)

In [3]:
import torch 
from sorl.gat import GATConfig, GAT

gat_config = GATConfig(vocab_sizes=[128,8],
          n_layer=12,
          n_head=6,
          n_embd=768,
          flex_kernel_options=None)

model = GAT(gat_config)


token_ids = torch.randint(0, 128 + 8, (2, 4))
idx = token_ids[:, :-1].contiguous()
target = token_ids[:, 1:].contiguous()


# forward pass ()
ppt = model(idx, target, 1024)

In [4]:
from sorl.gat import parallel_denoise
from sorl.gat import generate 


num_iterations = 5 
memory_span = 1024 
temperature = 0.0

parallel_denoise(model, idx, num_iterations=5, memory_span=1024, temperature=0.0)

generate(model, idx, max_new_tokens=5, abstraction_interval=3)


tensor([[ 42,  86, 110, 129,   0,   0, 129,   0],
        [ 46,  21,   4, 129,   0,   0, 129,   0]])

In [24]:
idx = torch.tensor([
    [4, 129, 129, 129],
    [3, 10, 129, 129]
])

In [1]:
import torch 
from sorl.gat_act import GATConfig, GAT

gat_config = GATConfig(vocab_sizes=[128,8],
          n_layer=12,
          n_head=6,
          n_embd=768,
          flex_kernel_options=None)

model = GAT(gat_config)

from sorl.gat_act import infer_level

# idx = torch.randint(0, model.vocab_sizes.sum(), (2, 4)).contiguous()
idx = torch.tensor([
    [4, 129, 130, 129],
    [3, 10, 130, 129]
]).contiguous()
levels = infer_level(idx, model.vocab_sizes)
abstract_mask = (levels > 0)

In [4]:
from sorl.gat_act import search, generate

# search(model, idx, max_iterations=1, n_continuous=0, memory_span=1024, temperature=0.0, K=3)

# Issue 1. generate method keeps producing abstract tokens
generate(model, idx, max_new_tokens=10, K=3)

tensor([[  4, 129, 129, 129,   0,   0, 129,   0,   0, 129,   0,   0, 129,   0],
        [  3,  10, 129, 129,   0,   0, 129,   0,   0, 129,   0,   0, 129,   0]])